# **UNIVERSIDADE FEDERAL DO CEARA**
---
Disciplina: Introducao a analise em Big Data

---

Professor: Luiz Alexandre

---

Alunos:
1.   Julio Cesar Gama Feitosa Freitas - 583956
2.   Vitoria Freire Rocha Teixeira de Oliveira - 587661

---
Data: 13/09/2026

# 🧪 Lab 11 — Dashboard: visualizar para decidir

## 🎯 Objetivo

Montar um dashboard com os 4 blocos do slide "Anatomia" (DIA 3, slide 10): KPIs, tendência, composição, detalhe.


In [17]:
# Instala o Plotly (pandas ja vem pre-instalado no Colab)
!pip install plotly --quiet

In [18]:
# Importa as bibliotecas e cria as pastas utilizadas pelo laboratorio
import os
import shutil
import duckdb

os.makedirs("bigdata/raw/customers", exist_ok=True)
os.makedirs("bigdata/raw/transactions", exist_ok=True)

# Abre uma conexao DuckDB (usada so para reconstruir Bronze/Silver/Gold)
con = duckdb.connect()

print("Ambiente preparado.")

Ambiente preparado.


## Passo 0 - Reconstruir Bronze, Silver e Gold

In [19]:
# Faz o upload dos CSVs brutos: customers_synthetic.csv e transactions_synthetic.csv
# from google.colab import files

uploaded = [
    "../customers_synthetic.csv",
    "../transactions_synthetic.csv",
    "../fraud_labels.csv"
]

missing = [f for f in uploaded if not os.path.exists(f)] # Verifica se todos os arquivos necessários foram carregados
if missing:
    raise FileNotFoundError("Arquivos ausentes: " + ", ".join(missing))

print("\n✓ Os 3 datasets foram encontrados.")


✓ Os 3 datasets foram encontrados.


In [20]:
# Copia os CSVs enviados para a estrutura Raw do projeto
for name in uploaded:
    if "customers_synthetic" in name:
        shutil.copy(name, "bigdata/raw/customers/customers_synthetic.csv")
    elif "transactions_synthetic" in name:
        shutil.copy(name, "bigdata/raw/transactions/transactions_synthetic.csv")

print("Arquivos Raw preparados.")

Arquivos Raw preparados.


In [21]:
# Recria a Bronze de clientes e de transacoes com as mesmas regras do Lab 6
customers_raw = "bigdata/raw/customers/customers_synthetic.csv"
transactions_raw = "bigdata/raw/transactions/transactions_synthetic.csv"

con.sql(f"""
CREATE OR REPLACE TABLE bronze_customers AS
SELECT DISTINCT
    customer_id, name, cpf, email, segment,
    CAST(credit_score AS INT) AS credit_score,
    CAST(created_at AS DATE) AS created_at
FROM read_csv_auto('{customers_raw}')
WHERE customer_id IS NOT NULL
  AND credit_score BETWEEN 300 AND 900
""")

# O CASE converte explicitamente True/False (texto) para BOOLEAN
con.sql(f"""
CREATE OR REPLACE TABLE bronze_transactions AS
SELECT DISTINCT
    transaction_id, customer_id,
    CAST(amount AS FLOAT) AS amount,
    transaction_type, status,
    CAST(risk_score AS FLOAT) AS risk_score,
    CASE WHEN is_fraud = 'True' THEN true ELSE false END AS is_fraud,
    CAST(timestamp AS TIMESTAMP) AS ts
FROM read_csv_auto('{transactions_raw}')
WHERE amount > 0
  AND customer_id IS NOT NULL
""")

con.sql("SELECT COUNT(*) AS total FROM bronze_customers").show()
con.sql("SELECT COUNT(*) AS total FROM bronze_transactions").show()

┌───────┐
│ total │
│ int64 │
├───────┤
│  9993 │
└───────┘

┌────────┐
│ total  │
│ int64  │
├────────┤
│ 100000 │
└────────┘



In [22]:
# Recria a Silver juntando transacoes com clientes e derivando as colunas de analise
con.sql("""
CREATE OR REPLACE TABLE silver_transactions AS
SELECT
  t.transaction_id, t.customer_id, t.amount, t.transaction_type,
  t.status, t.risk_score, t.is_fraud, t.ts,
  c.segment, c.credit_score,
  year(t.ts)  AS year,
  month(t.ts) AS month,
  day(t.ts)   AS day,
  dayofweek(t.ts) AS day_of_week,
  CASE
    WHEN t.amount < 100  THEN 'baixo'
    WHEN t.amount < 1000 THEN 'medio'
    ELSE 'alto'
  END AS amount_band
FROM bronze_transactions t
JOIN bronze_customers c ON t.customer_id = c.customer_id
""")

con.sql("SELECT COUNT(*) FROM silver_transactions").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       100000 │
└──────────────┘



In [23]:
# Recria a Gold gold_fraud_risk com as mesmas regras do Lab 7 - e a fonte do dashboard
con.sql("""
CREATE OR REPLACE TABLE gold_fraud_risk AS
SELECT
  segment,
  COUNT(*) AS total_transacoes,
  SUM(amount) AS valor_total,
  ROUND(AVG(amount), 2) AS ticket_medio,
  SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS qtd_fraudes,
  ROUND(100.0 * SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) / COUNT(*), 2) AS taxa_fraude_pct,
  SUM(CASE WHEN is_fraud THEN amount ELSE 0 END) AS valor_em_risco
FROM silver_transactions
GROUP BY segment
""")

con.sql("SELECT * FROM gold_fraud_risk ORDER BY taxa_fraude_pct DESC").show()

┌───────────┬──────────────────┬────────────────────┬──────────────┬─────────────┬─────────────────┬────────────────────┐
│  segment  │ total_transacoes │    valor_total     │ ticket_medio │ qtd_fraudes │ taxa_fraude_pct │   valor_em_risco   │
│  varchar  │      int64       │       double       │    double    │   int128    │     double      │       double       │
├───────────┼──────────────────┼────────────────────┼──────────────┼─────────────┼─────────────────┼────────────────────┤
│ High-Risk │             9155 │ 1664640.2447309494 │       181.83 │         705 │             7.7 │ 126451.06910800934 │
│ Standard  │            29689 │  5479399.846734047 │       184.56 │         655 │            2.21 │ 106828.41910934448 │
│ Premium   │            61156 │ 11235041.576210976 │       183.71 │         473 │            0.77 │  81308.43799591064 │
└───────────┴──────────────────┴────────────────────┴──────────────┴─────────────┴─────────────────┴────────────────────┘



In [24]:
# Exporta a Gold para CSV, exatamente como no Passo do Lab 9 (Opcao 1)
con.sql("COPY gold_fraud_risk TO 'fraud_risk_export.csv' (HEADER, DELIMITER ',')")

print("fraud_risk_export.csv gerado.")

fraud_risk_export.csv gerado.


## Passo 1 - Carregar os dados exportados

In [25]:
# Le o CSV recem-exportado, do mesmo jeito que o Lab 11 original espera
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fraud_risk = pd.read_csv('fraud_risk_export.csv')
print(fraud_risk)

     segment  total_transacoes   valor_total  ticket_medio  qtd_fraudes  \
0   Standard             29689  5.479400e+06        184.56          655   
1    Premium             61156  1.123504e+07        183.71          473   
2  High-Risk              9155  1.664640e+06        181.83          705   

   taxa_fraude_pct  valor_em_risco  
0             2.21   106828.419109  
1             0.77    81308.437996  
2             7.70   126451.069108  


## Passo 2 - Montar as 4 secoes do template de dashboard

In [26]:
# Cria a grade 2x2: 2 indicadores (KPIs) na linha de cima,
# grafico de barras e tabela de detalhe na linha de baixo
fig = make_subplots(
    rows=2, cols=2,
    specs=[[{"type":"indicator"}, {"type":"indicator"}],
           [{"type":"bar"}, {"type":"table"}]],
    subplot_titles=("", "", "Taxa de fraude por segmento", "Detalhe por segmento")
)

# 1) KPI - total de transacoes (soma dos 3 segmentos)
fig.add_trace(go.Indicator(
    mode="number", value=fraud_risk['total_transacoes'].sum(),
    title={"text": "Total de Transacoes"}
), row=1, col=1)

# 2) KPI - taxa de fraude geral (fraudes totais / transacoes totais, nao a media dos 3 segmentos)
taxa_geral = 100 * fraud_risk['qtd_fraudes'].sum() / fraud_risk['total_transacoes'].sum()
fig.add_trace(go.Indicator(
    mode="number", value=round(taxa_geral, 2),
    number={"suffix": "%"},
    title={"text": "Taxa de Fraude Geral"}
), row=1, col=2)

# 3) Composicao - taxa de fraude por segmento em barras
fig.add_trace(go.Bar(
    x=fraud_risk['segment'], y=fraud_risk['taxa_fraude_pct'],
    marker_color=['#2ecc71','#f5a623','#e5484d']
), row=2, col=1)

# 4) Detalhe - todas as colunas da Gold em formato de tabela navegavel
fig.add_trace(go.Table(
    header=dict(values=list(fraud_risk.columns)),
    cells=dict(values=[fraud_risk[c] for c in fraud_risk.columns])
), row=2, col=2)

fig.update_layout(height=650, title_text="Dashboard - Risco de Fraude", showlegend=False)
fig.write_html("dashboard_fraude.html")
print("Salvo em dashboard_fraude.html")

Salvo em dashboard_fraude.html


## Passo 3 - Adicionar interatividade (hover)

In [27]:
# Customiza o hover do grafico de barras para mostrar o valor exato com 2 casas decimais
fig.update_traces(hovertemplate="%{x}: %{y:.2f}%", row=2, col=1)
fig.write_html("dashboard_fraude.html")

print("dashboard_fraude.html atualizado - passe o mouse na barra para ver o valor exato.")

dashboard_fraude.html atualizado - passe o mouse na barra para ver o valor exato.


## Passo 4 - Baixar e abrir o dashboard

In [28]:
# Baixa o HTML para o computador local - abra o arquivo direto no navegador
# from google.colab import files as colab_files

# colab_files.download("dashboard_fraude.html")

## Checkpoint

- [ ] Dashboard tem os 4 blocos: KPI, KPI, composicao, detalhe
- [ ] Arquivo `dashboard_fraude.html` abre no navegador e o hover mostra valores
- [ ] Voce consegue apontar, em 1 frase, o que cada grafico responde

---

**Proximo lab:** `DIA3_LAB12_ML_PREVIEW.md` - o ultimo: um modelo simples de fraude.